# 04_search_demo
Query the FAISS index with new images and visualize results.

In [1]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import torch

# Ensure the project root is on PYTHONPATH so local src/ imports work in notebook
project_root = os.path.abspath(os.path.join('..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.clip_embedding import load_clip_model, load_image
from src.vector_search import load_faiss_index, search_index

metadata = pd.read_csv('../data/metadata.csv')
index = load_faiss_index('../data/faiss_index.faiss')
model, processor, device = load_clip_model()

def query_image(image_path, top_k=5):
    img = load_image(image_path)
    inputs = processor(images=img, return_tensors='pt').to(device)

    with torch.no_grad():
        image_outputs = model.get_image_features(**inputs)
        if hasattr(image_outputs, 'pooler_output') and image_outputs.pooler_output is not None:
            vec_tensor = image_outputs.pooler_output
        elif isinstance(image_outputs, (tuple, list)) and len(image_outputs) > 0:
            vec_tensor = image_outputs[0]
        else:
            vec_tensor = image_outputs

        if not torch.is_tensor(vec_tensor):
            raise TypeError(f'Unexpected query vector type: {type(vec_tensor)}')

        vec = torch.nn.functional.normalize(vec_tensor, p=2, dim=1).cpu().numpy()[0]

    distances, indices = search_index(index, vec, top_k=top_k)
    return distances, indices

# Select a sample query image from metadata for reliable demo.
query_path = 'F:/25-26 kì 2/chuyên đề thầy Thành/TrafficSignsProject/data_test/Yield_sign_in_Monaco.jpg'
if query_path is None or not os.path.isfile(query_path):
    raise FileNotFoundError('No valid sample image found in metadata. Please run 01_prepare_dataset first.')

print('Using query image:', query_path)
distances, indices = query_image(query_path, top_k=5)
print('Distances', distances)
for score, idx in zip(distances, indices):
    print(score, metadata.iloc[idx]['label'], metadata.iloc[idx]['group'])

c:\Users\xuann\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 398/398 [00:01<00:00, 391.76it/s, Materializing param=visual_projection.weight]                                
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using query image: F:/25-26 kì 2/chuyên đề thầy Thành/TrafficSignsProject/data_test/Yield_sign_in_Monaco.jpg
Distances [0.9265671  0.9205759  0.9204852  0.9197334  0.91618085]
0.9265671 W.215b nguy hiểm
0.9205759 W.215a nguy hiểm
0.9204852 W.215b nguy hiểm
0.9197334 W.215c nguy hiểm
0.91618085 W.215c nguy hiểm
